# Craigslist ask: five-fold CatBoost AFT + TF–IDF → UMAP

Run on Kaggle with **Internet on**. CatBoost `SurvivalAft` runs on CPU; switching on GPU does not accelerate this loss. The original Craigslist archive is checked by SHA256 before use. Each fold fits its own TF–IDF and UMAP on training descriptions only. The saved artifacts include fold models, fold text transformers, OOF residuals, and USD MAE. Asking prices are advertised amounts, not confirmed sales.


In [ ]:
!pip -q install 'numpy>=1.26,<3' 'pandas>=2.1,<3' 'scikit-learn>=1.4,<2' 'scipy>=1.11,<2' 'catboost==1.2.10' 'lightgbm>=4.5,<5' 'requests>=2.31,<3' 'umap-learn>=0.5.7,<0.6' 'joblib>=1.3,<2'


In [ ]:
import os, subprocess
from pathlib import Path
root=Path('/kaggle/working/Pricing')
if not root.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/botsandbolts/Pricing.git',str(root)],check=True)
else: subprocess.run(['git','-C',str(root),'pull','--ff-only'],check=True)
os.chdir(root)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
import hashlib, urllib.request
archive=Path('training-cache/ask.zip');archive.parent.mkdir(exist_ok=True)
if not archive.exists(): urllib.request.urlretrieve('https://zenodo.org/records/10457828/files/Used%20Cars%20Dataset.zip?download=1',archive)
digest=hashlib.sha256(archive.read_bytes()).hexdigest()
assert digest=='eb2ad6c7dedd3c73ffed4bf47929ab66e55d7da186ed240230caa1fad85d2e8f',digest
print('Original Craigslist archive verified')


In [ ]:
subprocess.run(['python','-m','scripts.prepare_full',str(archive),'--output','training-cache/full-cohort.jsonl','--workers','4'],check=True)


In [ ]:
subprocess.run(['python','-m','scripts.train_catboost_fivefold','--market','ask','--ask','training-cache/full-cohort.jsonl','--iterations','1000','--output','/kaggle/working/catboost-fivefold'],check=True)


In [ ]:
import json, pandas as pd
report=json.loads(Path('/kaggle/working/catboost-fivefold/ask/evaluation.json').read_text())
print(json.dumps(report,indent=2))


In [ ]:
import shutil
shutil.make_archive('/kaggle/working/catboost-fivefold-ask','zip','/kaggle/working/catboost-fivefold','ask')
print('Download /kaggle/working/catboost-fivefold-ask.zip from Kaggle Output')
